

# 📈 Forecasting & Anomaly Detection

<div align="center">
  <img src="https://miro.medium.com/v2/resize:fit:640/format:webp/1*2fD9qRIlRAJay7D5NK3yEA.png" width="200"><span></span><img src="./line-dot-graph.png" width="200">
</div>

### **Overview**
This template provides an end-to-end time-series analysis and outlier detection pipeline designed to run on **Saturn Cloud Jupyter Notebooks**. Optimized for **CPU execution**, it demonstrates an ensemble approach using **Prophet** for seasonality and **Orbit** for Bayesian structural modeling. By leveraging the scalable environment of **Saturn Cloud**, this workflow can handle complex Bayesian sampling (MAP estimation) with high efficiency.

### **Dataset Overview**
The template utilizes existing time-series data (Log-scale Wikipedia page views), which is a standard benchmark for evaluating forecasting accuracy.

### **Tech Stack**
* **Python**: Core language for time-series logic.
* **Prophet (Meta)**: Decomposable model for handling seasonality and holidays.
* **Orbit (Uber)**: Bayesian Damped Local Trend (DLT) framework.
* **Infrastructure**: [Saturn Cloud](https://saturncloud.io/) CPU-based Jupyter Instance.

In [ ]:
# Install core forecasting and Bayesian libraries
!pip install prophet orbit-ml pandas matplotlib -q

### **Step 1: Load Existing Data**
We ingest historical time-series data and prepare it for the modeling engines.

In [ ]:
import pandas as pd
url = "https://raw.githubusercontent.com/facebook/prophet/main/examples/example_wp_log_peyton_manning.csv"
df = pd.read_csv(url)
df['ds'] = pd.to_datetime(df['ds'])
df.head()

### **Step 2: Bayesian Trend Modeling (Orbit)**
We use Uber's **Orbit** library to fit a Damped Local Trend (DLT) model.

In [ ]:
from orbit.models import DLT
model_orbit = DLT(response_col='y', date_col='ds', seasonality=52)
model_orbit.fit(df)
print("Orbit Bayesian model training complete.")

### **Step 3: Anomaly Detection (Prophet)**
Identifying historical data points that deviate from the 95% confidence interval.

In [ ]:
from prophet import Prophet
import matplotlib.pyplot as plt
m = Prophet(interval_width=0.95)
m.fit(df)
forecast = m.predict(df)
performance = pd.merge(df, forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']], on='ds')
performance['anomaly'] = (performance['y'] > performance['yhat_upper']) | (performance['y'] < performance['yhat_lower'])
anomalies = performance[performance['anomaly'] == True]

### **Step 4: Final Visualization**
Visualizing the anomaly alerts in a unified chart.

In [ ]:
plt.figure(figsize=(14, 7))
plt.plot(performance['ds'], performance['y'], color='black', label='Actual', alpha=0.5)
plt.fill_between(performance['ds'], performance['yhat_lower'], performance['yhat_upper'], color='blue', alpha=0.2, label='Confidence Interval')
plt.scatter(anomalies['ds'], anomalies['y'], color='red', label='Anomaly Alert', s=10)
plt.title("Forecasting Result")
plt.legend()
plt.show()

---
## 🏁 Conclusion & Next Steps
This template demonstrates how Bayesian structural models can be successfully deployed for anomaly detection. For a verifiable and reachable production environment, you can scale this notebook into a scheduled job or an API endpoint using the **Saturn Cloud** platform.

### **Resources & Backlinks**
* **Cloud Infrastructure**: [Deploy this on Saturn Cloud](https://saturncloud.io/)
* **Orbit Library**: [Bayesian DLT Documentation](https://orbit-ml.readthedocs.io/)
* **Prophet Guide**: [Meta's Forecasting Docs](https://facebook.github.io/prophet/)